# SpiderNet tutorial: HGSOC model training and molecular concordance

See [README.md](README.md) for the execution order, upstream inputs, commands and paper mapping.
Run the unified entry point to save executed copies and local outputs. Scientific variants and their parameters are retained below.


## Before you start

Use a Python environment with the local SpiderNet package and its tutorial
dependencies installed (including PyTorch, PyTorch Geometric, torch-scatter,
Scanpy, NumPy, SciPy, pandas, and Matplotlib). Select that environment as the
Jupyter kernel; the notebook assumes `SpiderNet` is already importable.

Set `PROCESSED_DATA_DIR` to the bundle produced by
`spidernet_dataloading_MIdimselection_HGSOC.ipynb` and set `OUTPUT_ROOT` for this run.
The configured paths are local examples and must point to the intended data and
model outputs. The file check below lists the required bundle contents.
`metadata_sample.csv` is required by the optional cell-type-pair analysis.

Run the training sections in order. Model initialization/training can be
stochastic; this notebook does not set a training seed. Preserve the published
checkpoint and its matching processed bundle when reproducing a published run.
An existing `Model/model_epoch19999.pth` is loaded for the default 20,000 epochs;
otherwise training starts and writes checkpoints to the configured run directory.


## 0. Dataset setup

Configure input and output paths and the training hyperparameters. The default
15 MIs and 20,000 epochs correspond to the HGSOC configuration described in the
manuscript. The setup JSON is consumed by the downstream HGSOC notebooks.


In [ ]:
from pathlib import Path
import json

# Processed data folder
from workflow_paths import PROCESSED_DATA_DIR, RESULTS_ROOT, input_path, output_path, ensure_output
ensure_output()

# Output folder for training results
OUTPUT_ROOT = RESULTS_ROOT

SPECIES = "human"

# Training hyperparameters
DIM_ENVIR = 15
N_JOBS = 5
MAX_EPOCH = 20000

VERSION = "V1"

training_setup = {
    "PROCESSED_DATA_DIR": str(PROCESSED_DATA_DIR),
    "OUTPUT_ROOT": str(OUTPUT_ROOT),
    "SPECIES": SPECIES,
    "DIM_ENVIR": DIM_ENVIR,
    "N_JOBS": N_JOBS,
    "MAX_EPOCH": MAX_EPOCH,
    "VERSION": VERSION,
}

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
setup_path = OUTPUT_ROOT / "HGSOC_modeltraining_setup.json"
with open(setup_path, "w", encoding="utf-8") as f:
    json.dump(training_setup, f, indent=2)

print(f"Training setup saved to: {setup_path}")
training_setup


## 1. Imports and device setup

In [ ]:

import json
from pathlib import Path

import pandas as pd
import scanpy as sc
import torch

from SpiderNet.api import (
    build_model,
    run_training,
    infer_meta_interactions,
    normalize_outputs,
    export_results,
)
from SpiderNet.config import PathConfig, TrainingConfig
from SpiderNet.io import load_processed_data

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


## 2. Check the processed data folder

Verify that the required files for training are present.


In [ ]:

required_training_files = [
    "SpiderNet_data_pyg_list.pkl",
    "LR_list.pkl",
    "LR_list_cellchatdb.pkl",
    "LR_meta_cellchatdb.pkl",
    "batch_cell_unique.pkl",
    "batch_cell.pkl",
    "genenames_train.pkl",
    "adata_list.pkl",
    "cellclass_unique.pkl",
    "adata_all.h5ad",
]

recommended_hgsoc_files = [
    "metadata_sample.csv",
    "bundle_summary.json",
]

rows = []
for name in required_training_files:
    path = PROCESSED_DATA_DIR / name
    rows.append({
        "group": "required_for_training",
        "file": name,
        "path": str(path),
        "exists": input_path(path).exists(),
    })

for name in recommended_hgsoc_files:
    path = PROCESSED_DATA_DIR / name
    rows.append({
        "group": "recommended_for_hgsoc_downstream",
        "file": name,
        "path": str(path),
        "exists": input_path(path).exists(),
    })

status_df = pd.DataFrame(rows)
display(status_df)

missing_required = status_df.loc[
    (status_df["group"] == "required_for_training") & (~status_df["exists"]),
    "file",
].tolist()

if missing_required:
    raise FileNotFoundError(
        "The processed bundle is incomplete. Missing required files: "
        + ", ".join(missing_required)
    )

bundle_summary_path = PROCESSED_DATA_DIR / "bundle_summary.json"
if input_path(bundle_summary_path).exists():
    with open(input_path(bundle_summary_path), "r", encoding="utf-8") as f:
        bundle_summary = json.load(f)
    print("Found bundle_summary.json")
    display(pd.Series(bundle_summary, name="value"))
else:
    bundle_summary = None
    print("bundle_summary.json not found. Training can still proceed.")

adata_all_path = PROCESSED_DATA_DIR / "adata_all.h5ad"
adata_example = sc.read_h5ad(input_path(adata_all_path), backed="r")
overview = pd.Series({
    "processed_dir": str(PROCESSED_DATA_DIR),
    "n_cells_total": int(adata_example.n_obs),
    "n_genes_total": int(adata_example.n_vars),
    "obs_columns": list(adata_example.obs.columns),
    "obsm_keys": list(adata_example.obsm.keys()),
}, name="value")
display(overview.to_frame())
adata_example.file.close()


## 3. Build training config and run directories

In [ ]:
train_cfg = TrainingConfig(
    dim_envir=DIM_ENVIR,
    n_jobs=N_JOBS,
    max_epoch=MAX_EPOCH,
    version=VERSION,
)

paths = PathConfig(
    data_root=PROCESSED_DATA_DIR,
    output_root=OUTPUT_ROOT,
    species=SPECIES,
    version=VERSION,
)

run_dirs = paths.ensure_dirs(dim_envir=train_cfg.dim_envir)
print(run_dirs)

with open(paths.output_root / "run_dirs.json", "w", encoding="utf-8") as handle:
    json.dump({k: str(v) for k, v in run_dirs.items()}, handle, indent=2)

train_cfg

# Keep checkpoint reuse upstream and write analysis exports locally.
from workflow_paths import LOCAL_RUN
run_dirs["run_dir"] = LOCAL_RUN


## 4. Load the processed SpiderNet objects

In [ ]:

processed = load_processed_data(PROCESSED_DATA_DIR)

print("Processed directory:", PROCESSED_DATA_DIR)
print("Number of batches:", len(processed.spidernet_data))
print("Number of LR pairs:", len(processed.lr_list))
print("Number of training genes:", processed.genenames_train.shape[0])


Inspect batch dimensions before training. The edge check below compares the
largest stored endpoint with the number of cells; it is a diagnostic and does
not establish complete graph validity or detect all isolated cells.


In [ ]:

summary_rows = []
for i in range(len(processed.adata_list)):
    adata_i = processed.adata_list[i]
    edge_index_max = torch.max(processed.spidernet_data[i]["edge_index"]).cpu().item()
    summary_rows.append(
        {
            "batch_index": i,
            "n_cells": adata_i.n_obs,
            "n_genes": adata_i.n_vars,
            "edge_index_max": edge_index_max,
            "edge_index_matches_n_cells": adata_i.n_obs == (edge_index_max + 1),
        }
    )

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

issues = []
if len(processed.spidernet_data) == 0:
    issues.append("No processed batches were loaded.")
if len(processed.lr_list) == 0:
    issues.append("No ligand-receptor pairs were retained.")
if processed.genenames_train.shape[0] == 0:
    issues.append("No training genes were retained.")
if not summary_df["edge_index_matches_n_cells"].all():
    issues.append("At least one batch has an edge index / cell-count mismatch.")

if issues:
    print("Sanity check flagged the following issues:")
    for issue in issues:
        print("-", issue)
else:
    print("Sanity checks passed. The processed data appear internally consistent.")


## 5. Build and train the SpiderNet model

In [ ]:

model = build_model(
    processed=processed,
    train_cfg=train_cfg,
    device=device,
)

print(model)


Run training, or load the final checkpoint if it already exists in the model
directory. Keep checkpoints paired with the processed bundle that generated them;
checkpoint existence alone does not verify data provenance.


In [ ]:

model = run_training(
    model=model,
    processed=processed,
    train_cfg=train_cfg,
    model_dir=run_dirs["model_dir"],
    device=device,
)


## 6. Infer MI activities and molecular loadings

Infer MI activities in processed-batch and directed-edge order. Normalize each
MI activity column by its global maximum and rescale its LR, sender-regulator,
and receiver-target loadings correspondingly. Export occurs in Section 7.


In [ ]:

results = infer_meta_interactions(model=model, processed=processed)
results = normalize_outputs(results)

print("Factor_envir shape:", results["factor_envir"].shape)
print("LR loading shape:", results["loading_lr"].shape)
print("Receiver loading shape:", results["loading_receiver"].shape)
print("Sender loading shape:", results["loading_sender"].shape)


### Export one slice for the realistic simulation

Save normalized MI activities, processed AnnData, and directed edges from batch
index 0 as inputs to the realistic simulation workflow. The chosen slice and
its outputs are retained separately from the full HGSOC run.


In [ ]:
Slice_index_choose = 0
import numpy as np

In [ ]:
MI_strength_chooseslice = results['factor_envir_list'][Slice_index_choose]
# Export edge-level activities in the selected slice's stored edge order.
np.save(run_dirs["run_dir"] / f"MI_strength_slice{Slice_index_choose}.npy", MI_strength_chooseslice)


In [ ]:
adata_chooseslice = processed.adata_list[Slice_index_choose]
# Export the matching processed cells.
adata_chooseslice.write_h5ad(run_dirs["run_dir"] / f"adata_slice{Slice_index_choose}.h5ad")


In [ ]:
adata_chooseslice.obsm['spatial']

In [ ]:
edgeindex_chooseslice = processed.spidernet_data[Slice_index_choose]['edge_index'].cpu().numpy()
# Columns are sender and receiver indices into the selected AnnData.
np.save(run_dirs["run_dir"] / f"edgeindex_slice{Slice_index_choose}.npy", edgeindex_chooseslice)


## 7. Save training outputs

In [ ]:

export_results(
    results=results,
    processed=processed,
    precessed_data_dir=PROCESSED_DATA_DIR,
    output_dir=run_dirs["run_dir"],
)

with open(run_dirs["model_dir"] / "SpiderNet_model_config.json", "w", encoding="utf-8") as handle:
    json.dump(train_cfg.to_dict(), handle, indent=2)

provenance = {
    "processed_data_dir": str(PROCESSED_DATA_DIR),
    "output_root": str(OUTPUT_ROOT),
    "run_dir": str(run_dirs["run_dir"]),
    "model_dir": str(run_dirs["model_dir"]),
    "species": SPECIES,
    "dim_envir": DIM_ENVIR,
    "n_jobs": N_JOBS,
    "max_epoch": MAX_EPOCH,
    "version": VERSION,
}

if bundle_summary is not None:
    provenance["processed_bundle_summary_path"] = str(bundle_summary_path)

with open(run_dirs["run_dir"] / "SpiderNet_training_provenance.json", "w", encoding="utf-8") as handle:
    json.dump(provenance, handle, indent=2)

print(f"Results saved to: {run_dirs['run_dir']}")
provenance


## Key output files

Training results are saved under `run_dirs["run_dir"]`:

- `Factor_envir_use.npy` and `Factor_envir_list.pkl`: normalized directed-edge MI activities, stacked and per batch.
- `loading_LR_use.npy`, `loading_sender_use.npy`, and `loading_receiver_use.npy`: MI-by-feature loadings.
- `loading_sender_use.csv`, `loading_receiver_use.csv`, and `loading_intrinsic_use.csv`: labeled loading tables.
- `MI_strength_slice0.npy`, `adata_slice0.h5ad`, and `edgeindex_slice0.npy`: simulation inputs for the selected slice.
- `SpiderNet_training_provenance.json`: configured data/run provenance.

Checkpoints and `SpiderNet_model_config.json` are stored in `run_dirs["model_dir"]`.
The optional sections add MI correlations, pathway/cell-type-pair summaries,
Figure S9a loading heatmaps and matrices, and Figure S9b concordance outputs.


## Optional downstream analyses

These sections use the processed bundle and the matching exported model results.
Run Optional 2 before Optional 3: the cell-type-pair analysis reads the exported
`LR_loading_pathway.csv`. Optional 4 uses its MI order when that file is available.

For Figure S9b alone, complete dataset/configuration setup and load `processed`
(Sections 0-4), then run Optional 5 against an existing exported run. Optional 5
does not require the enrichment sections or the heatmap helpers. Its correlations
use the observed model inputs, rather than reconstructed gene expression.


## Optional 1. Correlations between MI activity dimensions

Summarize relationships among MI dimensions with the existing `MI_correlation`
helper. This is distinct from the MI-to-molecular-component analysis in Figure S9b.


In [ ]:

from SpiderNet.analysis import MI_correlation

mi_results = MI_correlation(
    run_dirs["run_dir"],
    input_path(run_dirs["run_dir"] / "Factor_envir_use.npy"),
    show=True,
)


## Optional 2. LR loading-based pathway enrichment

In [ ]:

from SpiderNet.analysis import LRLoading_enrichment

LRLoading_enrichment(
    loading_LR_use_path=input_path(run_dirs["run_dir"] / "loading_LR_use.npy"),
    lr_list_path=input_path(PROCESSED_DATA_DIR / "LR_list.pkl"),
    lr_list_cellchatdb_path=input_path(PROCESSED_DATA_DIR / "LR_list_cellchatdb.pkl"),
    lr_meta_cellchatdb_path=input_path(PROCESSED_DATA_DIR / "LR_meta_cellchatdb.pkl"),
    Factor_envir_use_path=input_path(run_dirs["run_dir"] / "Factor_envir_use.npy"),
    file_savepath_main=run_dirs["run_dir"],
    show=True,
)


## Optional 3. Sender-receiver cell-type-pair summaries

Summarize MI activities by ordered cell-type pair and relate them to LR pathways.
This section requires sample metadata and Optional 2's pathway output. The
configured `MIlevel_agg_threshold=0.6` is retained. Figure 3b's caption instead
describes `>0.7` for highlighting malignant-involving pairs, so this setting
requires reconciliation before claiming exact reproduction of that panel.


In [ ]:

from SpiderNet.analysis import MI_Celltypepair_enrichment

MI_Celltypepair_enrichment(
    SpiderNet_data_pyg_list_path=input_path(PROCESSED_DATA_DIR / "SpiderNet_data_pyg_list.pkl"),
    Factor_envir_use_path=input_path(run_dirs["run_dir"] / "Factor_envir_use.npy"),
    file_savepath_main=run_dirs["run_dir"],
    metadata_sample_path=input_path(PROCESSED_DATA_DIR / "metadata_sample.csv"),
    LR_loading_pathway_path=input_path(run_dirs["run_dir"] / "LR_loading_pathway.csv"),
    dim_envir=train_cfg.dim_envir,
    MIlevel_agg_threshold=0.6,
    batch_cell_unique_path=input_path(PROCESSED_DATA_DIR / "batch_cell_unique.pkl"),
    adata_list_path=input_path(PROCESSED_DATA_DIR / "adata_list.pkl"),
    adata_copy_path=input_path(PROCESSED_DATA_DIR / "adata_all.h5ad"),
    show=True,
)


## Optional 4. Molecular loading programs (Figure S9a)

Plot sender-regulator, LR-pair, and receiver-target loadings after feature-wise
sum normalization across MIs. Features are grouped by their dominant MI, with
the MI order taken from `LR_loading_pathway.csv` when available. The exported
sorted matrices record the heatmap data. This section retains its existing
normalization and color limits.

### Heatmap configuration and helpers


In [ ]:
# ============================================================
# Combined MI loading heatmaps:
#   1) Sender regulator loading: white -> #134E8E
#   2) LR-pair loading:          white -> #FF4400
#   3) Receiver target loading:  white -> #C00707
#
# Each matrix is feature-wise sum-normalized:
#   value[MI, feature] = loading[MI, feature] / sum_MI loading[MI, feature]
#
# Columns within each heatmap are sorted by the MI dimension where
# each gene/LR pair reaches its maximum normalized loading.
# ============================================================

import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec


# -----------------------------
# User-adjustable options
# -----------------------------
USE_LR_PATHWAY_MI_ORDER = True
PREFER_RAW_NPY = True
DROP_ALL_ZERO_FEATURES = True

SHOW_XTICK_LABELS = False
MAX_XTICKS_TO_SHOW = 80

VMAX_QUANTILE = 0.995
SHOW = True

SAVE_PREFIX = "MI_loading_heatmaps_sender_LR_receiver_sum_normalized_combined"

out_dir = Path(run_dirs["run_dir"])
out_dir.mkdir(parents=True, exist_ok=True)


# -----------------------------
# Helper functions
# -----------------------------
def _canonical_mi_name(x, fallback_index=None):
    """
    Convert MI labels into canonical MI-1 style.
    """
    s = str(x).strip()

    if s.startswith("MI-"):
        return s

    if s.startswith("MI"):
        digits = "".join([c for c in s if c.isdigit()])
        if digits != "":
            return f"MI-{int(digits)}"

    if fallback_index is not None:
        return f"MI-{fallback_index + 1}"

    digits = "".join([c for c in s if c.isdigit()])
    if digits != "":
        return f"MI-{int(digits)}"

    return s


def _natural_mi_order(mi_names):
    """
    Sort MI names by numeric MI index.
    """
    def key_fun(x):
        digits = "".join([c for c in str(x) if c.isdigit()])
        return int(digits) if digits != "" else 10**9

    return sorted(list(mi_names), key=key_fun)


def _load_pickle(path):
    path = Path(path)
    with open(input_path(path), "rb") as f:
        return pickle.load(f)


def _format_lr_pair(lr_pair):
    """
    Convert LR pair object [[ligands], [receptors]] into readable string.
    """
    ligand, receptor = lr_pair

    if isinstance(ligand, str):
        ligand_str = ligand
    else:
        ligand_str = "+".join([str(x) for x in ligand])

    if isinstance(receptor, str):
        receptor_str = receptor
    else:
        receptor_str = "+".join([str(x) for x in receptor])

    return f"{ligand_str} -> {receptor_str}"


def _load_gene_names():
    """
    Load gene names used by the SpiderNet model.
    """
    candidate_paths = [
        PROCESSED_DATA_DIR / "genenames_train.pkl",
        PROCESSED_DATA_DIR / "genenames.pkl",
        PROCESSED_DATA_DIR / "gene_names.pkl",
    ]

    for path in candidate_paths:
        if input_path(Path(path)).exists():
            gene_names = _load_pickle(path)
            return list(np.asarray(gene_names).astype(str))

    raise FileNotFoundError(
        "Cannot find gene name file. Tried:\n"
        + "\n".join([str(x) for x in candidate_paths])
    )


def _load_lr_names():
    """
    Load LR-pair names.
    Prefer LR_list_merge.pkl generated by LRLoading_enrichment.
    """
    lr_merge_path = out_dir / "LR_list_merge.pkl"
    if input_path(lr_merge_path).exists():
        lr_names = _load_pickle(lr_merge_path)
        return list(np.asarray(lr_names).astype(str))

    lr_list_path = PROCESSED_DATA_DIR / "LR_list.pkl"
    if input_path(lr_list_path).exists():
        lr_list = _load_pickle(lr_list_path)
        return [_format_lr_pair(x) for x in lr_list]

    raise FileNotFoundError(
        "Cannot find LR-pair name file. Tried:\n"
        f"{lr_merge_path}\n{lr_list_path}"
    )


def _load_loading_matrix(raw_npy_path, csv_path=None, feature_names=None, dim_envir=None):
    """
    Load loading matrix as DataFrame with rows = MIs, columns = features.
    Prefer raw .npy if available; fallback to CSV.
    """
    raw_npy_path = Path(raw_npy_path)

    use_raw = PREFER_RAW_NPY and input_path(raw_npy_path).exists()

    if use_raw:
        arr = np.load(input_path(raw_npy_path))
        arr = np.asarray(arr, dtype=float)

        if dim_envir is None:
            dim_envir = arr.shape[0]

        # Ensure rows = MIs
        if arr.shape[0] != dim_envir and arr.shape[1] == dim_envir:
            arr = arr.T

        if arr.shape[0] != dim_envir:
            print(
                f"[Warning] {raw_npy_path.name}: expected {dim_envir} MI rows, "
                f"but got shape {arr.shape}."
            )

        if feature_names is None:
            feature_names = [f"Feature_{i}" for i in range(arr.shape[1])]
        else:
            feature_names = list(feature_names)

        if len(feature_names) != arr.shape[1]:
            print(
                f"[Warning] feature_names length ({len(feature_names)}) does not match "
                f"matrix columns ({arr.shape[1]}) for {raw_npy_path.name}. "
                "Using generic feature names."
            )
            feature_names = [f"Feature_{i}" for i in range(arr.shape[1])]

        mi_names = [f"MI-{i + 1}" for i in range(arr.shape[0])]
        df = pd.DataFrame(arr, index=mi_names, columns=feature_names)

    else:
        if csv_path is None:
            raise FileNotFoundError(f"Missing raw npy file: {raw_npy_path}")

        csv_path = Path(csv_path)
        if not input_path(csv_path).exists():
            raise FileNotFoundError(
                f"Missing both raw npy and csv files:\n"
                f"  npy: {raw_npy_path}\n"
                f"  csv: {csv_path}"
            )

        df = pd.read_csv(input_path(csv_path), index_col=0)
        df = df.apply(pd.to_numeric, errors="coerce").fillna(0)

        # If columns are MIs and rows are features, transpose.
        if dim_envir is not None and df.shape[1] == dim_envir and df.shape[0] != dim_envir:
            df = df.T

        df.index = [
            _canonical_mi_name(x, fallback_index=i)
            for i, x in enumerate(df.index)
        ]

    # Canonicalize MI labels
    df.index = [
        _canonical_mi_name(x, fallback_index=i)
        for i, x in enumerate(df.index)
    ]

    # Remove duplicated feature names if any
    if df.columns.duplicated().any():
        new_cols = []
        seen = {}
        for c in df.columns:
            if c not in seen:
                seen[c] = 0
                new_cols.append(c)
            else:
                seen[c] += 1
                new_cols.append(f"{c}__dup{seen[c]}")
        df.columns = new_cols

    return df


def _feature_sum_normalize(df, eps=1e-12):
    """
    Normalize each feature column so its sum across MIs is 1.
    """
    df = df.copy().astype(float)
    df[df < 0] = 0

    col_sum = df.sum(axis=0)

    if DROP_ALL_ZERO_FEATURES:
        keep_cols = col_sum > eps
        df = df.loc[:, keep_cols]
        col_sum = col_sum.loc[keep_cols]

    norm_df = df.div(col_sum.replace(0, np.nan), axis=1).fillna(0)
    return norm_df


def _get_mi_order(all_mi_names):
    """
    Use LR_loading_pathway.csv row order if available; otherwise natural MI order.
    """
    all_mi_names = list(all_mi_names)

    if USE_LR_PATHWAY_MI_ORDER:
        mi_order_path = out_dir / "LR_loading_pathway.csv"
        if input_path(mi_order_path).exists():
            order_df = pd.read_csv(input_path(mi_order_path), index_col=0)
            order_from_file = [
                _canonical_mi_name(x, fallback_index=i)
                for i, x in enumerate(order_df.index)
            ]
            order_from_file = [x for x in order_from_file if x in all_mi_names]
            remaining = [x for x in _natural_mi_order(all_mi_names) if x not in order_from_file]
            return order_from_file + remaining

    return _natural_mi_order(all_mi_names)


def _sort_columns_by_argmax_mi(norm_df, mi_order):
    """
    Sort feature columns by:
      1) MI where feature has max normalized loading
      2) max normalized loading descending within that MI
      3) feature name
    """
    mi_order = [m for m in mi_order if m in norm_df.index]
    norm_df = norm_df.loc[mi_order, :]

    values = norm_df.values
    max_pos = np.argmax(values, axis=0)
    max_val = np.max(values, axis=0)

    # All-zero columns go to the end
    group_rank = np.where(max_val > 0, max_pos, len(mi_order))

    feature_names = np.asarray(norm_df.columns.astype(str))
    col_order = np.lexsort((feature_names, -max_val, group_rank))

    sorted_df = norm_df.iloc[:, col_order]

    sorted_group_rank = group_rank[col_order]
    sorted_max_val = max_val[col_order]

    block_labels = []
    for g, v in zip(sorted_group_rank, sorted_max_val):
        if g >= len(mi_order) or v <= 0:
            block_labels.append("All-zero")
        else:
            block_labels.append(mi_order[int(g)])

    return sorted_df, block_labels


def _prepare_loading_heatmap(raw_df):
    """
    Normalize and sort one loading matrix.
    """
    norm_df = _feature_sum_normalize(raw_df)
    mi_order = _get_mi_order(norm_df.index)
    norm_sorted_df, block_labels = _sort_columns_by_argmax_mi(norm_df, mi_order)
    return norm_sorted_df, block_labels


def _make_white_to_color_cmap(hex_color, name):
    return LinearSegmentedColormap.from_list(
        name,
        ["#FFFFFF", hex_color],
        N=256,
    )


def _draw_one_heatmap(ax, cax, df, block_labels, title, cmap, show_ylabel=True):
    """
    Draw one heatmap into given axis and its horizontal colorbar axis.
    Use pcolormesh without cell edges to avoid white horizontal hairlines.
    """
    data = df.values
    n_mi, n_feat = data.shape

    vmax = np.nanquantile(data, VMAX_QUANTILE) if np.isfinite(data).any() else 1
    vmax = max(vmax, 1e-6)
    vmax = min(vmax, 1.0)

    # Important: avoid white row hairlines in PDF/viewer rendering
    x = np.arange(n_feat + 1)
    y = np.arange(n_mi + 1)

    im = ax.pcolormesh(
        x,
        y,
        data,
        cmap=cmap,
        vmin=0,
        vmax=vmax,
        shading="flat",
        edgecolors="none",
        linewidth=0,
        antialiased=False,
        rasterized=True,
    )

    ax.set_xlim(0, n_feat)
    ax.set_ylim(0, n_mi)
    ax.invert_yaxis()
    ax.grid(False)
    ax.set_facecolor("white")

    # Y-axis
    ax.set_yticks(np.arange(n_mi) + 0.5)
    if show_ylabel:
        ax.set_yticklabels(df.index, fontsize=10)
        ax.set_ylabel("Meta-interaction (MI)", fontsize=11)
    else:
        ax.set_yticklabels([])
        ax.set_ylabel("")

    # X-axis
    ax.set_xlabel("Features sorted by argmax MI", fontsize=10)

    if SHOW_XTICK_LABELS and n_feat <= MAX_XTICKS_TO_SHOW:
        ax.set_xticks(np.arange(n_feat) + 0.5)
        ax.set_xticklabels(df.columns, rotation=60, ha="right", fontsize=6)
    else:
        ax.set_xticks([])

    ax.set_title(title, fontsize=12, pad=26)

    # Vertical separators and top block labels
    block_labels = np.asarray(block_labels)
    unique_blocks = []
    for b in block_labels:
        if len(unique_blocks) == 0 or unique_blocks[-1] != b:
            unique_blocks.append(b)

    for b in unique_blocks:
        idx = np.where(block_labels == b)[0]
        if len(idx) == 0:
            continue

        start = idx.min()
        end = idx.max() + 1

        # Only draw vertical block separators, no horizontal separators
        ax.axvline(start, color="#B6B9BA", linewidth=0.45)

        if b != "All-zero":
            center = (start + end) / 2
            ax.text(
                center,
                -0.95,
                b,
                ha="center",
                va="bottom",
                fontsize=7,
                rotation=0,
                clip_on=False,
            )

    ax.axvline(n_feat, color="#B6B9BA", linewidth=0.45)

    # Remove extra spines
    for spine in ax.spines.values():
        spine.set_linewidth(0.6)

    # Horizontal colorbar below heatmap
    cb = plt.colorbar(im, cax=cax, orientation="horizontal")
    cb.set_label("Feature-wise sum-normalized loading", fontsize=9)
    cb.ax.tick_params(labelsize=8)

    return im


### Load, normalize, and plot the molecular loadings


In [ ]:
# -----------------------------
# Load feature names
# -----------------------------
gene_names = _load_gene_names()
lr_names = _load_lr_names()


# -----------------------------
# Load raw loading matrices
# -----------------------------
loading_sender_df = _load_loading_matrix(
    raw_npy_path=input_path(out_dir / "loading_sender_use.npy"),
    csv_path=input_path(out_dir / "loading_sender_use.csv"),
    feature_names=gene_names,
    dim_envir=train_cfg.dim_envir,
)

loading_lr_df = _load_loading_matrix(
    raw_npy_path=input_path(out_dir / "loading_LR_use.npy"),
    csv_path=input_path(out_dir / "loading_LR_use.csv"),
    feature_names=lr_names,
    dim_envir=train_cfg.dim_envir,
)

loading_receiver_df = _load_loading_matrix(
    raw_npy_path=input_path(out_dir / "loading_receiver_use.npy"),
    csv_path=input_path(out_dir / "loading_receiver_use.csv"),
    feature_names=gene_names,
    dim_envir=train_cfg.dim_envir,
)


print("Raw sender regulator loading shape:", loading_sender_df.shape)
print("Raw LR loading shape:", loading_lr_df.shape)
print("Raw receiver target loading shape:", loading_receiver_df.shape)


# -----------------------------
# Normalize and sort each heatmap independently
# -----------------------------
sender_regulator_loading_norm_sorted, sender_block_labels = _prepare_loading_heatmap(
    loading_sender_df
)

lr_loading_norm_sorted, lr_block_labels = _prepare_loading_heatmap(
    loading_lr_df
)

receiver_target_loading_norm_sorted, receiver_block_labels = _prepare_loading_heatmap(
    loading_receiver_df
)


print("Sorted sender regulator loading shape:", sender_regulator_loading_norm_sorted.shape)
print("Sorted LR loading shape:", lr_loading_norm_sorted.shape)
print("Sorted receiver target loading shape:", receiver_target_loading_norm_sorted.shape)


# -----------------------------
# Save sorted normalized matrices
# -----------------------------
sender_csv = out_dir / f"{SAVE_PREFIX}_sender_regulator_matrix.csv"
lr_csv = out_dir / f"{SAVE_PREFIX}_LR_pair_matrix.csv"
receiver_csv = out_dir / f"{SAVE_PREFIX}_receiver_target_matrix.csv"

sender_regulator_loading_norm_sorted.to_csv(sender_csv)
lr_loading_norm_sorted.to_csv(lr_csv)
receiver_target_loading_norm_sorted.to_csv(receiver_csv)

print(f"[Saved] {sender_csv}")
print(f"[Saved] {lr_csv}")
print(f"[Saved] {receiver_csv}")


# -----------------------------
# Plot combined horizontal figure
# -----------------------------
sender_cmap = _make_white_to_color_cmap("#134E8E", "sender_white_blue")
lr_cmap = _make_white_to_color_cmap("#FF4400", "lr_white_orange")
receiver_cmap = _make_white_to_color_cmap("#C00707", "receiver_white_red")

n_sender = sender_regulator_loading_norm_sorted.shape[1]
n_lr = lr_loading_norm_sorted.shape[1]
n_receiver = receiver_target_loading_norm_sorted.shape[1]

# Retained alternative: scale panel widths by feature counts instead of equal widths.
# width_ratios = np.array([
#     max(1.0, np.sqrt(n_sender)),
#     max(1.0, np.sqrt(n_lr)),
#     max(1.0, np.sqrt(n_receiver)),
# ])
# width_ratios = width_ratios / width_ratios.min()

width_ratios=[1, 1, 1]

fig_width = 22
fig_height = max(5.0, 0.32 * train_cfg.dim_envir + 2.1)

plt.close()
fig = plt.figure(figsize=(fig_width, fig_height))

gs = GridSpec(
    nrows=2,
    ncols=3,
    figure=fig,
    height_ratios=[18, 1.2],
    width_ratios=width_ratios,
    hspace=0.38,
    wspace=0.18,
)

ax_sender = fig.add_subplot(gs[0, 0])
ax_lr = fig.add_subplot(gs[0, 1])
ax_receiver = fig.add_subplot(gs[0, 2])

cax_sender = fig.add_subplot(gs[1, 0])
cax_lr = fig.add_subplot(gs[1, 1])
cax_receiver = fig.add_subplot(gs[1, 2])

_draw_one_heatmap(
    ax=ax_sender,
    cax=cax_sender,
    df=sender_regulator_loading_norm_sorted,
    block_labels=sender_block_labels,
    title="Sender regulator loading",
    cmap=sender_cmap,
    show_ylabel=True,
)

_draw_one_heatmap(
    ax=ax_lr,
    cax=cax_lr,
    df=lr_loading_norm_sorted,
    block_labels=lr_block_labels,
    title="LR-pair loading",
    cmap=lr_cmap,
    show_ylabel=False,
)

_draw_one_heatmap(
    ax=ax_receiver,
    cax=cax_receiver,
    df=receiver_target_loading_norm_sorted,
    block_labels=receiver_block_labels,
    title="Receiver target loading",
    cmap=receiver_cmap,
    show_ylabel=False,
)

fig.suptitle(
    "MI loading programs across sender regulators, LR bridges, and receiver targets",
    fontsize=14,
    y=0.995,
)

pdf_path = out_dir / f"{SAVE_PREFIX}.pdf"
png_path = out_dir / f"{SAVE_PREFIX}.png"

fig.savefig(pdf_path, bbox_inches="tight", dpi=300)
fig.savefig(png_path, bbox_inches="tight", dpi=300)

if SHOW:
    plt.show()

plt.close(fig)

print(f"[Saved] {pdf_path}")
print(f"[Saved] {png_path}")


# -----------------------------
# Optional sanity check
# -----------------------------
print(
    "Column-sum check after feature-wise normalization:",
    {
        "sender_minmax": (
            float(sender_regulator_loading_norm_sorted.sum(axis=0).min()),
            float(sender_regulator_loading_norm_sorted.sum(axis=0).max()),
        ),
        "LR_minmax": (
            float(lr_loading_norm_sorted.sum(axis=0).min()),
            float(lr_loading_norm_sorted.sum(axis=0).max()),
        ),
        "receiver_minmax": (
            float(receiver_target_loading_norm_sorted.sum(axis=0).min()),
            float(receiver_target_loading_norm_sorted.sum(axis=0).max()),
        ),
    },
)


In [ ]:
pdf_path

## Optional 5. Concordance between MI activity and its molecular components (Figure S9b)

For each MI, calculate Spearman correlations between (left) regulator-gene
expression and mean sending MI activity, (middle) LR-pair co-expression and
directed-edge MI activity, and (right) target-gene expression and mean receiving
MI activity. Pool all processed cells or edges across batches without sampling,
cell-type filtering, or a positive-expression filter.

Use `batch.x` and `batch.cellpair_LRpair_neigh` directly so the observed features
retain the preprocessing, complex-subunit handling, and ordering used for model
training. The LR matrix was computed as the geometric mean of sender ligand and
receiver receptor expression (Methods Eq. 1); complexes use the geometric mean
of their measured subunits. Sending and receiving activities are means over
stored outgoing and incoming edges (Eq. 5). Zero-degree cells follow the model's
zero-activity convention and are counted in the batch audit table.

Features are ordered by descending loadings normalized across MIs using Eq. 10
(`loading / (sum across MIs + 1e-6)`). The caption does not specify a tie rule or
rank scaling. Here, equal loadings share a descending dense rank, mapped to [0, 1]
as `(rank - 1) / (number of distinct loadings - 1)`. This places tied zero loadings
at the right edge, consistent with the reference panel's appearance; it is an
explicit reconstruction convention, not a documented original plotting rule.
If all loadings are tied, their coordinate is 0. The caption also does not state
whether correlations were pooled or summarized by sample; pooling is explicit here.

Retain all features, including zero loadings and negative correlations. Expression
and activity ties use average ranks for Spearman's statistic. Constant variables
yield undefined correlations (NaN), recorded in tables and omitted from visible
points. Red points indicate **rho > 0.4**, not statistical significance; no
P-value or multiple-testing procedure is specified for this panel.

The standalone helper cell and execution cell below add a figure (PDF/PNG), a
complete feature-level CSV, an MI summary, a batch alignment/degree audit, and
settings JSON in `run_dirs["run_dir"]/MI_component_concordance/`. Feature chunking
limits temporary memory without changing the set of observations. Dimensions
are checked, but the inputs must still come from the same run: matching shapes
cannot detect a same-sized reordered or unrelated processed bundle.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata


def _concordance_numpy(value):
    """Read a tensor or array without modifying the stored training inputs."""
    if hasattr(value, "detach"):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def _concordance_mean_activity(edge_mi, cell_index, n_cells):
    """Average directed edge activities by endpoint, as in Methods Eq. 5."""
    degree = np.bincount(cell_index, minlength=n_cells)
    activity = np.zeros((n_cells, edge_mi.shape[1]), dtype=np.float64)
    for mi in range(edge_mi.shape[1]):
        activity[:, mi] = np.bincount(
            cell_index, weights=edge_mi[:, mi], minlength=n_cells
        )
    # Match the model's scatter_mean convention for cells with no such edges.
    np.divide(activity, degree[:, None], out=activity, where=degree[:, None] > 0)
    return activity, degree


def _concordance_spearman(feature_batches, activity, feature_chunk_size=8):
    """Compute pooled Spearman rho with average ranks for expression/activity ties.

    Rank each feature over all observations, not separately within each batch.
    Chunking features bounds temporary memory without subsampling observations.
    Constant variables have undefined correlations and remain NaN.
    """
    n_obs, n_mi = activity.shape
    n_features = feature_batches[0].shape[1]
    if n_obs < 2 or sum(x.shape[0] for x in feature_batches) != n_obs:
        raise ValueError("Features and MI activities must share at least two observations.")
    if feature_chunk_size < 1:
        raise ValueError("feature_chunk_size must be positive.")
    if any(x.ndim != 2 or x.shape[1] != n_features for x in feature_batches):
        raise ValueError("Feature columns must have the same order in every batch.")
    if not np.isfinite(activity).all():
        raise ValueError("MI activities contain non-finite values.")

    activity_ranks = rankdata(activity, axis=0, method="average")
    activity_ranks -= activity_ranks.mean(axis=0)
    activity_norm = np.sqrt(np.einsum("ij,ij->j", activity_ranks, activity_ranks))
    rho = np.full((n_mi, n_features), np.nan, dtype=np.float64)
    for start in range(0, n_features, feature_chunk_size):
        stop = min(start + feature_chunk_size, n_features)
        values = np.concatenate([x[:, start:stop] for x in feature_batches], axis=0)
        if not np.isfinite(values).all():
            raise ValueError("Molecular features contain non-finite values.")
        feature_ranks = rankdata(values, axis=0, method="average")
        del values
        feature_ranks -= feature_ranks.mean(axis=0)
        feature_norm = np.sqrt(np.einsum("ij,ij->j", feature_ranks, feature_ranks))
        denominator = activity_norm[:, None] * feature_norm[None, :]
        np.divide(
            activity_ranks.T @ feature_ranks,
            denominator,
            out=rho[:, start:stop],
            where=denominator > 0,
        )
    return rho


def _concordance_table(component, rho, loadings, feature_names, n_observations):
    """Order features by MI-specific loadings normalized as in Methods Eq. 10."""
    loadings = np.asarray(loadings, dtype=np.float64)
    if loadings.shape != rho.shape or loadings.shape[1] != len(feature_names):
        raise ValueError(f"Loading/feature dimensions do not match for {component}.")
    if not np.isfinite(loadings).all() or np.any(loadings < 0):
        raise ValueError(f"Expected finite non-negative {component} loadings.")
    normalized = loadings / (loadings.sum(axis=0, keepdims=True) + 1e-6)
    frames = []
    for mi in range(loadings.shape[0]):
        # Equal loadings share an x coordinate; zero-loading features remain visible.
        dense_rank = rankdata(-normalized[mi], method="dense")
        rank_01 = (dense_rank - 1) / max(int(dense_rank.max()) - 1, 1)
        frame = pd.DataFrame({
            "component": component,
            "mi": f"MI-{mi + 1}",
            "feature_index": np.arange(len(feature_names)),
            "feature": list(feature_names),
            "loading": loadings[mi],
            "normalized_loading": normalized[mi],
            "loading_dense_rank": dense_rank,
            "loading_rank_01": rank_01,
            "spearman_rho": rho[mi],
            "rho_gt_0p4": rho[mi] > 0.4,
            "n_observations": n_observations,
        })
        frames.append(frame.sort_values("loading_rank_01", kind="stable"))
    return pd.concat(frames, ignore_index=True)


def compute_mi_component_concordance(
    spidernet_data, edge_activity, loading_sender, loading_lr, loading_receiver,
    gene_names, lr_names, feature_chunk_size=8,
):
    """Calculate Figure S9b from aligned processed batches and one trained run.

    Each stored edge_index has shape (n_edges, 2): sender then receiver.
    edge_activity must follow inference/export batch order and within-batch edge order.
    The stored x and cellpair_LRpair_neigh are the observed reconstruction inputs.
    """
    if not spidernet_data:
        raise ValueError("No processed batches were supplied.")
    edge_activity = np.asarray(edge_activity)
    if edge_activity.ndim != 2 or edge_activity.shape[1] == 0:
        raise ValueError("edge_activity must have shape (n_edges, n_mi).")
    expected_shapes = [
        (loading_sender, len(gene_names)), (loading_lr, len(lr_names)),
        (loading_receiver, len(gene_names)),
    ]
    if any(np.shape(w) != (edge_activity.shape[1], n) for w, n in expected_shapes):
        raise ValueError("MI dimensions and feature names must match the saved loadings.")

    expression_batches, lr_batches = [], []
    sending_batches, receiving_batches, batch_rows = [], [], []
    offset = 0
    for batch_index, batch in enumerate(spidernet_data):
        expression = _concordance_numpy(batch["x"])
        edges = _concordance_numpy(batch["edge_index"])
        lr_coexpression = _concordance_numpy(batch["cellpair_LRpair_neigh"])
        if expression.ndim != 2 or expression.shape[1] != len(gene_names):
            raise ValueError(f"Training gene columns do not match in batch {batch_index}.")
        if edges.ndim != 2 or edges.shape[1] != 2:
            raise ValueError("SpiderNet edge_index must have shape (n_edges, 2).")
        if not np.issubdtype(edges.dtype, np.integer):
            raise ValueError("Edge endpoints must be integer cell indices.")
        n_cells, n_edges = expression.shape[0], edges.shape[0]
        if edges.size and (edges.min() < 0 or edges.max() >= n_cells):
            raise ValueError(f"Invalid edge endpoint in batch {batch_index}.")
        if lr_coexpression.shape != (n_edges, len(lr_names)):
            raise ValueError(f"LR co-expression is not aligned in batch {batch_index}.")
        if offset + n_edges > len(edge_activity):
            raise ValueError("Saved MI activities have fewer rows than the processed edges.")
        batch_mi = edge_activity[offset:offset + n_edges]
        sending, out_degree = _concordance_mean_activity(batch_mi, edges[:, 0], n_cells)
        receiving, in_degree = _concordance_mean_activity(batch_mi, edges[:, 1], n_cells)
        expression_batches.append(expression)
        lr_batches.append(lr_coexpression)
        sending_batches.append(sending)
        receiving_batches.append(receiving)
        batch_rows.append({
            "batch_index": batch_index,
            "n_cells": n_cells,
            "n_edges": n_edges,
            "edge_row_start": offset,
            "edge_row_stop": offset + n_edges,
            "n_cells_without_outgoing_edges": int((out_degree == 0).sum()),
            "n_cells_without_incoming_edges": int((in_degree == 0).sum()),
        })
        offset += n_edges
    if offset != len(edge_activity):
        raise ValueError("Saved MI activities have more rows than the processed edges.")

    component_tables = []
    components = [
        ("sender_regulator", expression_batches, np.vstack(sending_batches), loading_sender, gene_names),
        ("LR_pair", lr_batches, edge_activity, loading_lr, lr_names),
        ("receiver_target", expression_batches, np.vstack(receiving_batches), loading_receiver, gene_names),
    ]
    for component, feature_batches, activity, loadings, names in components:
        print(f"Computing {component}: {len(activity):,} observations, {len(names):,} features", flush=True)
        rho = _concordance_spearman(feature_batches, activity, feature_chunk_size)
        component_tables.append(_concordance_table(component, rho, loadings, names, len(activity)))
    return pd.concat(component_tables, ignore_index=True), pd.DataFrame(batch_rows)


def plot_mi_component_concordance(concordance_table):
    """Plot three loading-ranked correlation profiles per MI, as in Figure S9b."""
    panels = [
        ("sender_regulator", "Gene expression vs. sending MI activity", "Regulatory gene rank"),
        ("LR_pair", "LR co-expression vs. MI activity", "Ligand-receptor pair rank"),
        ("receiver_target", "Gene expression vs. receiving MI activity", "Target gene rank"),
    ]
    mi_names = sorted(concordance_table["mi"].unique(), key=lambda s: int(s.split("-")[1]))
    finite_rho = concordance_table.loc[
        np.isfinite(concordance_table["spearman_rho"]), "spearman_rho"
    ]
    # Include negative correlations and all observed positive values without clipping.
    ymin = min(-0.2, float(finite_rho.min()) - 0.04) if len(finite_rho) else -0.2
    ymax = max(0.8, float(finite_rho.max()) + 0.04) if len(finite_rho) else 0.8
    with plt.rc_context({"font.size": 8, "axes.grid": False, "pdf.fonttype": 42}):
        fig, axes = plt.subplots(
            len(mi_names), 3, figsize=(14, max(3.0, 0.43 * len(mi_names) + 0.9)),
            squeeze=False, sharex=True, sharey=True,
        )
        fig.subplots_adjust(left=0.065, right=0.945, bottom=0.09, top=0.93, hspace=0.12, wspace=0.15)
        for row, mi in enumerate(mi_names):
            for col, (component, title, xlabel) in enumerate(panels):
                ax = axes[row, col]
                data = concordance_table.loc[
                    (concordance_table["mi"] == mi)
                    & (concordance_table["component"] == component)
                ].sort_values("loading_rank_01", kind="stable")
                x = data["loading_rank_01"].to_numpy()
                y = data["spearman_rho"].to_numpy()
                ax.plot(x, y, color="#666666", linewidth=0.3, marker=".", markersize=1.4)
                highlight = np.isfinite(y) & (y > 0.4)
                ax.scatter(x[highlight], y[highlight], color="#FF3333", s=4, linewidths=0, zorder=3)
                ax.set_xlim(-0.015, 1.015)
                ax.set_ylim(ymin, ymax)
                ax.set_yticks([0.0, 0.4])
                ax.tick_params(axis="y", labelleft=True, labelsize=7, length=2, pad=1)
                ax.set_xticks([0.0, 0.5, 1.0])
                for side in ("top", "right"):
                    ax.spines[side].set_visible(False)
                ax.spines["left"].set_linewidth(0.5)
                ax.spines["bottom"].set_visible(row == len(mi_names) - 1)
                ax.tick_params(axis="x", bottom=row == len(mi_names) - 1, labelsize=8, length=2)
                if row == 0:
                    ax.set_title(title, fontsize=10, pad=9)
                if row == len(mi_names) - 1:
                    ax.set_xlabel(xlabel, fontsize=9)
                if col == 2:
                    ax.text(1.025, 0.5, mi, transform=ax.transAxes, va="center", fontsize=8)
        fig.text(0.015, 0.51, "Spearman ρ", rotation=90, va="center", fontsize=10)
        fig.text(0.015, 0.975, "b", fontsize=12, fontweight="bold")
    return fig


### Calculate concordance and export Figure S9b


In [ ]:
# Run after loading processed and exporting the matching trained model results.
# Use canonical feature order from the processed bundle, not a heatmap-sorted CSV.
CONCORDANCE_FEATURE_CHUNK_SIZE = 8
CONCORDANCE_SHOW = True
concordance_dir = Path(run_dirs["run_dir"]) / "MI_component_concordance"
concordance_dir.mkdir(parents=True, exist_ok=True)
concordance_gene_names = list(np.asarray(processed.genenames_train).astype(str))
concordance_lr_names = [
    "+".join([ligand] if isinstance(ligand, str) else map(str, ligand))
    + " -> "
    + "+".join([receptor] if isinstance(receptor, str) else map(str, receptor))
    for ligand, receptor in processed.lr_list
]

concordance_table, concordance_batches = compute_mi_component_concordance(
    spidernet_data=processed.spidernet_data,
    edge_activity=np.load(input_path(Path(run_dirs["run_dir"]) / "Factor_envir_use.npy"), mmap_mode="r"),
    loading_sender=np.load(input_path(Path(run_dirs["run_dir"]) / "loading_sender_use.npy")),
    loading_lr=np.load(input_path(Path(run_dirs["run_dir"]) / "loading_LR_use.npy")),
    loading_receiver=np.load(input_path(Path(run_dirs["run_dir"]) / "loading_receiver_use.npy")),
    gene_names=concordance_gene_names,
    lr_names=concordance_lr_names,
    feature_chunk_size=CONCORDANCE_FEATURE_CHUNK_SIZE,
)
concordance_table.to_csv(concordance_dir / "MI_component_concordance.csv", index=False)
concordance_batches.to_csv(concordance_dir / "MI_component_concordance_batches.csv", index=False)
concordance_summary = concordance_table.groupby(["component", "mi"], sort=False).agg(
    n_features=("feature_index", "size"),
    n_defined_correlations=("spearman_rho", "count"),
    n_rho_gt_0p4=("rho_gt_0p4", "sum"),
    n_observations=("n_observations", "first"),
).reset_index()
concordance_summary.to_csv(concordance_dir / "MI_component_concordance_summary.csv", index=False)

concordance_settings = {
    "paper_panel": "Figure S9b",
    "expression": "stored processed PyG x; no further normalization",
    "LR_coexpression": "stored cellpair_LRpair_neigh; geometric mean across directed endpoints",
    "aggregation": "mean over outgoing edges for sending; incoming edges for receiving",
    "zero_degree_cells": "zero activity, matching model scatter_mean",
    "observations": "all processed cells / directed edges pooled in batch order; no subsampling",
    "statistic": "Spearman rho with average ranks for expression/activity ties",
    "constant_variables": "undefined rho retained as NaN; no zero imputation",
    "loading_normalization": "loading / (feature sum across MIs + 1e-6), Methods Eq. 10",
    "feature_order": "descending normalized loading; stable order within ties",
    "x_coordinate": "(descending dense rank - 1) / (number of distinct loadings - 1); all tied -> 0",
    "highlight": "rho > 0.4, strictly positive; not a P-value threshold",
    "n_batches": len(concordance_batches),
    "n_cells": int(concordance_batches["n_cells"].sum()),
    "n_edges": int(concordance_batches["n_edges"].sum()),
    "feature_chunk_size": CONCORDANCE_FEATURE_CHUNK_SIZE,
}
with open(concordance_dir / "MI_component_concordance_settings.json", "w", encoding="utf-8") as handle:
    json.dump(concordance_settings, handle, indent=2)

concordance_figure = plot_mi_component_concordance(concordance_table)
for extension in ("pdf", "png"):
    concordance_figure.savefig(
        concordance_dir / f"MI_component_concordance_S9b.{extension}",
        bbox_inches="tight", dpi=300,
    )
if CONCORDANCE_SHOW:
    plt.show()
plt.close(concordance_figure)
display(concordance_summary)
print(f"Concordance tables and Figure S9b saved to: {concordance_dir}")


## Reuse for other datasets

Update the processed-data and output directories, species, MI dimensionality,
training hyperparameters, and optional metadata inputs. Regenerate the processed
bundle before training, and keep its feature and edge order paired with the
exported model results. Figure S9b settings describe the HGSOC analysis; changing
the data or training run does not preserve the manuscript's MI identities.
